In [9]:
#!/usr/bin/env python3
"""
This script contains the cortical data used in manuscript 
NEUROTRANSMITTER TRANSPORTER/RECEPTOR CO-EXPRESSION SHARES ORGANIZATIONAL TRAITS WITH BRAIN STRUCTURE AND FUNCTION
https://doi.org/10.1101/2022.08.26.505274

Original code: https://github.com/CNG-LAB/cngopen/tree/main/receptor_similarity/code

Developed by Benjamin Hänisch

Adapted: Carlos Estevez-Fraga
"""


from netneurotools.networks import struct_consensus
from netneurotools.freesurfer import find_parcel_centroids
from nilearn.input_data import NiftiLabelsMasker
from nilearn import datasets
from nilearn._utils import check_niimg
import os
import pandas as pd
from scipy.stats import zscore
import re
import numpy as np


In [10]:
parcels = 100
input_path = '/Users/charlie/Desktop/my_projects/neurotransmitter/github/data/'

In [11]:
#pet data
#parcellate study maps, adapted from Hansen et al, https://doi.org/10.1101/2021.10.28.466336
atl='schaefer'
parc = 100
scale = '{}Parcels7Networks'.format(parc)

In [12]:
dataset=datasets.fetch_atlas_schaefer_2018(n_rois=parc, yeo_networks=7)
outpath=input_path + 'PET_nifti_images/'+"Parcellated/{}/{}".format(atl, scale)
if not os.path.exists(outpath):
    os.makedirs(outpath)


In [13]:
receptors_nii = [input_path+'PET_nifti_images/'+'5HT1a_way_hc36_savli.nii',
                 input_path+'PET_nifti_images/'+'5HT1a_cumi_hc8_beliveau.nii',
                 input_path+'PET_nifti_images/'+ '5HT1b_az_hc36_beliveau.nii',
                 input_path+'PET_nifti_images/'+'5HT1b_p943_hc22_savli.nii',
                 input_path+'PET_nifti_images/'+'5HT1b_p943_hc65_gallezot.nii.gz',
                 input_path+'PET_nifti_images/'+'5HT2a_cimbi_hc29_beliveau.nii',
                 input_path+'PET_nifti_images/'+'5HT2a_alt_hc19_savli.nii',
                 input_path+'PET_nifti_images/'+'5HT2a_mdl_hc3_talbot.nii.gz',
                 input_path+'PET_nifti_images/'+'5HT4_sb20_hc59_beliveau.nii',
                 input_path+'PET_nifti_images/'+'5HT6_gsk_hc30_radnakrishnan.nii.gz',
                 input_path+'PET_nifti_images/'+'5HTT_dasb_hc100_beliveau.nii',
                 input_path+'PET_nifti_images/'+'5HTT_dasb_hc30_savli.nii',
                 input_path+'PET_nifti_images/'+'A4B2_flubatine_hc30_hillmer.nii.gz',
                 input_path+'PET_nifti_images/'+'CB1_omar_hc77_normandin.nii.gz',
                 input_path+'PET_nifti_images/'+'CB1_FMPEPd2_hc22_laurikainen.nii',
                 input_path+'PET_nifti_images/'+'D1_SCH23390_hc13_kaller.nii',
                 input_path+'PET_nifti_images/'+'D2_fallypride_hc49_jaworska.nii',
                 input_path+'PET_nifti_images/'+'D2_flb457_hc37_smith.nii.gz',
                 input_path+'PET_nifti_images/'+'D2_flb457_hc55_sandiego.nii.gz',
                 input_path+'PET_nifti_images/'+'DAT_fpcit_hc174_dukart_spect.nii',
                 input_path+'PET_nifti_images/'+'DAT_fepe2i_hc6_sasaki.nii.gz',
                 input_path+'PET_nifti_images/'+'GABAa-bz_flumazenil_hc16_norgaard.nii',
                 input_path+'PET_nifti_images/'+'GABAa_flumazenil_hc6_dukart.nii',
                 input_path+'PET_nifti_images/'+'H3_cban_hc8_gallezot.nii.gz',
                 input_path+'PET_nifti_images/'+'M1_lsn_hc24_naganawa.nii.gz',
                 input_path+'PET_nifti_images/'+'mGluR5_abp_hc22_rosaneto.nii',
                 input_path+'PET_nifti_images/'+'mGluR5_abp_hc28_dubois.nii',
                 input_path+'PET_nifti_images/'+'mGluR5_abp_hc73_smart.nii',
                 input_path+'PET_nifti_images/'+'MU_carfentanil_hc204_kantonen.nii',
                 input_path+'PET_nifti_images/'+'MU_carfentanil_hc39_turtonen.nii',
                 input_path+'PET_nifti_images/'+'NAT_MRB_hc77_ding.nii.gz',
                 input_path+'PET_nifti_images/'+'NAT_MRB_hc10_hesse.nii',
                 input_path+'PET_nifti_images/'+'NMDA_ge179_hc29_galovic.nii.gz',
                 input_path+'PET_nifti_images/'+'VAChT_feobv_hc4_tuominen.nii',
                 input_path+'PET_nifti_images/'+'VAChT_feobv_hc5_bedard_sum.nii',
                 input_path+'PET_nifti_images/'+'VAChT_feobv_hc18_aghourian_sum.nii']


In [17]:

parcellated = {}
mask = NiftiLabelsMasker(dataset['maps'], resampling_target='data', strategy='mean').fit()

for receptor in receptors_nii:
    img = check_niimg(receptor, atleast_4d=True)
    parcellated[receptor] = mask.transform(img).squeeze()
    name = receptor.split('/')[-1]  # get nifti file name
    name = name.split('.')[0]  # remove .nii
    np.savetxt(outpath+'/'+ name+'.csv', parcellated[receptor], delimiter=',')

In [20]:

#generate comprehensive df
all_files = os.listdir(outpath)
#with open(input_path + 'NTRM_interest.txt', 'r') as f:
with open(input_path + 'receptor_list.txt', 'r') as f:
    s = f.read()
    s = s.split(',\n')
    studies = [re.search('.+?(?=\.)', study).group() for study in s]
receptors = {}
for study in studies:
    for i in all_files:
        if study in i:
            receptors[study] = i

In [21]:
# make dataframe from dict and returns output
vals={}
for key, val in receptors.items():
    #vals[key]=np.genfromtxt(outpath + val, delimiter=',')
    vals[key]=np.genfromtxt(outpath +'/' + val, delimiter=',')
df=pd.DataFrame(vals)
df.columns = [x + '.csv' for x in df.columns]

In [22]:
# generate weighted averages
_5HT1b = (zscore(df['5HT1b_p943_hc22_savli.csv']) * 22 +
          zscore(df['5HT1b_p943_hc65_gallezot.csv']) * 65) / (22 + 65)
_D2 = (zscore(df['D2_flb457_hc37_smith.csv']) * 37 +
       zscore(df['D2_flb457_hc55_sandiego.csv']) * 55) / (37 + 55)
_mGluR5 = (zscore(df['mGluR5_abp_hc22_rosaneto.csv']) * 22 +
           zscore(df['mGluR5_abp_hc28_dubois.csv']) * 28 +
           zscore(df['mGluR5_abp_hc73_smart.csv']) * 73) / (22 + 28 + 73)
_VAChT = (zscore(df['VAChT_feobv_hc18_aghourian_sum.csv']) * 18 +
          zscore(df['VAChT_feobv_hc4_tuominen.csv']) * 4 +
          zscore(df['VAChT_feobv_hc5_bedard_sum.csv'])* 5) / (18 + 4 + 5)


In [23]:
d = {'5HT1a': df['5HT1a_way_hc36_savli.csv'], '5HT1b': _5HT1b, '5HT2a': df['5HT2a_cimbi_hc29_beliveau.csv'],
     '5HT4': df['5HT4_sb20_hc59_beliveau.csv'], '5HT6': df['5HT6_gsk_hc30_radnakrishnan.csv'],
     '5HTT': df['5HTT_dasb_hc100_beliveau.csv'],
     'A4B2': df['A4B2_flubatine_hc30_hillmer.csv'], 'CB1': df['CB1_omar_hc77_normandin.csv'],
     'D1': df['D1_SCH23390_hc13_kaller.csv'], 'D2': _D2, 'DAT': df['DAT_fpcit_hc174_dukart_spect.csv'],
     'GABAa': df['GABAa-bz_flumazenil_hc16_norgaard.csv'], 'H3': df['H3_cban_hc8_gallezot.csv'],
     'M1': df['M1_lsn_hc24_naganawa.csv'], 'mGluR5': _mGluR5, 'MU': df['MU_carfentanil_hc204_kantonen.csv'],
     'NAT': df['NAT_MRB_hc77_ding.csv'],'NMDA' : df['NMDA_ge179_hc29_galovic.csv'], 'VAChT': _VAChT}


In [24]:
df_wm_reg = pd.DataFrame(d)
df_wm_reg.to_csv(input_path +'{}_receptorprofiles.csv'.format(scale))


In [26]:
lh='/Users/charlie/Parcellations/FreeSurfer5.3/fsaverage5/label/lh.Schaefer2018_100Parcels_7Networks_order.annot'
rh='/Users/charlie/Parcellations/FreeSurfer5.3/fsaverage5/label/lh.Schaefer2018_100Parcels_7Networks_order.annot'


In [27]:
centroids=find_parcel_centroids(lhannot=lh, rhannot=rh, version='fsaverage5')
points=centroids[0]
dist=np.empty((parc,parc))
for i in range(len(points)):
    for j in range(len(points)):
        dist[i,j]=np.linalg.norm(points[i]-points[j])


hemiid=np.concatenate((np.ones(int(parc / 2)),np.zeros(int(parc / 2))))
hemiid=hemiid.reshape(-1,1)
np.save(input_path + 'cort_dist_{}.npy'.format(parc), dist)